In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

from pathlib import Path
import pandas as pd
import numpy as np

folder = Path("/content/drive/MyDrive/Project2_SolarPV")

HORIZONS = {
    1: "15min",
    2: "30min",
    3: "45min",
    4: "60min",
}

Mounted at /content/drive


(CSV FILE data load + 34 DAYS SPLITED INTO 24/5/5 DAYS)

Load all four feature datasets

In [2]:
data = {}

for h in HORIZONS:
    input_path = folder / f"df_clean_h{h}.csv"

    if not input_path.is_file():
        raise FileNotFoundError(
            f"Missing {input_path}. Run the updated 02b notebook first."
        )

    d = pd.read_csv(
        input_path,
        parse_dates=["DATE_TIME", "TARGET_TIME"],
    )
    data[h] = d.sort_values("DATE_TIME").reset_index(drop=True)

    if not data[h]["DATE_TIME"].is_unique:
        raise ValueError(f"Duplicate feature timestamps found for h={h}.")

    print(f"h={h} ({HORIZONS[h]}): {len(data[h])} rows loaded")

h=1 (15min): 3113 rows loaded
h=2 (30min): 3105 rows loaded
h=3 (45min): 3098 rows loaded
h=4 (60min): 3092 rows loaded


##Define day-based splits
This creates one set of calendar-day boundaries from the 15-minute dataset and applies the same boundaries to every horizon.


In [3]:
days = np.array(
    sorted(data[1]["DATE_TIME"].dt.normalize().unique())
)

n_days = len(days)

if n_days != 34:
    print(f"Note: expected 34 days, found {n_days}.")

n_train = int(round(n_days * 0.70))
n_val = int(round(n_days * 0.15))

train_days = days[:n_train]
val_days = days[n_train:n_train + n_val]
test_days = days[n_train + n_val:]

print("Total days:", n_days)
print(
    f"Train: {len(train_days)} days "
    f"({pd.Timestamp(train_days[0]).date()} to "
    f"{pd.Timestamp(train_days[-1]).date()})"
)
print(
    f"Validation: {len(val_days)} days "
    f"({pd.Timestamp(val_days[0]).date()} to "
    f"{pd.Timestamp(val_days[-1]).date()})"
)
print(
    f"Test: {len(test_days)} days "
    f"({pd.Timestamp(test_days[0]).date()} to "
    f"{pd.Timestamp(test_days[-1]).date()})"
)

split_of_day = {}

for split_name, split_days in [
    ("train", train_days),
    ("val", val_days),
    ("test", test_days),
]:
    for day in split_days:
        split_of_day[pd.Timestamp(day)] = split_name

Total days: 34
Train: 24 days (2020-05-15 to 2020-06-07)
Validation: 5 days (2020-06-08 to 2020-06-12)
Test: 5 days (2020-06-13 to 2020-06-17)


##Apply splits and purge boundary-crossing targets
A row stays in the data only when its target timestamp falls within the same split as its feature timest

In [4]:
split_data = {}

for h, d in data.items():
    feature_split = d["DATE_TIME"].dt.normalize().map(split_of_day)
    target_split = d["TARGET_TIME"].dt.normalize().map(split_of_day)

    keep = (
        feature_split.notna()
        & target_split.notna()
        & feature_split.eq(target_split)
    )

    out = d.loc[keep].copy()
    out["SPLIT"] = feature_split.loc[keep].to_numpy()
    out = out.reset_index(drop=True)

    split_data[h] = out

    # Confirm no retained row has its target in a different split.
    assert (
        out["DATE_TIME"].dt.normalize().map(split_of_day).to_numpy()
        == out["TARGET_TIME"].dt.normalize().map(split_of_day).to_numpy()
    ).all(), f"h={h}: a target crosses a split boundary"

    print(
        f"h={h} ({HORIZONS[h]}): "
        f"{len(out)} rows kept; "
        f"{len(d) - len(out)} rows purged"
    )

h=1 (15min): 3111 rows kept; 2 rows purged
h=2 (30min): 3101 rows kept; 4 rows purged
h=3 (45min): 3092 rows kept; 6 rows purged
h=4 (60min): 3084 rows kept; 8 rows purged


Verify chronological order and split sizes

In [5]:
for h, d in split_data.items():
    train = d[d["SPLIT"] == "train"]
    val = d[d["SPLIT"] == "val"]
    test = d[d["SPLIT"] == "test"]

    assert len(train) > 0 and len(val) > 0 and len(test) > 0
    assert train["DATE_TIME"].max() < val["DATE_TIME"].min()
    assert val["DATE_TIME"].max() < test["DATE_TIME"].min()
    assert train["TARGET_TIME"].max() < val["DATE_TIME"].min()
    assert val["TARGET_TIME"].max() < test["DATE_TIME"].min()

    print(
        f"{HORIZONS[h]} — "
        f"train: {len(train)}, validation: {len(val)}, test: {len(test)}"
    )

print("All horizons passed the chronological and target-boundary checks.")

15min — train: 2159, validation: 479, test: 473
30min — train: 2152, validation: 478, test: 471
45min — train: 2145, validation: 477, test: 470
60min — train: 2139, validation: 476, test: 469
All horizons passed the chronological and target-boundary checks.


##Summarize daily irradiation by split
This uses the available 15-minute feature rows to summarize insolation for

In [6]:
d1 = split_data[1].copy()
d1["date"] = d1["DATE_TIME"].dt.normalize()

daily = (
    d1.groupby("date")
    .agg(
        split=("SPLIT", "first"),
        insolation_kWh_m2=("IRRADIATION", lambda values: values.sum() * 0.25),
        peak_irradiation=("IRRADIATION", "max"),
    )
)

print(daily.round(2).to_string())
print("\nInsolation summary by split:")
display(
    daily.groupby("split")["insolation_kWh_m2"]
    .agg(["mean", "min", "max"])
    .round(2)
)

            split  insolation_kWh_m2  peak_irradiation
date                                                  
2020-05-15  train               4.76              0.89
2020-05-16  train               4.66              0.81
2020-05-17  train               5.73              1.00
2020-05-18  train               3.82              0.97
2020-05-19  train               3.87              0.81
2020-05-20  train               4.26              0.98
2020-05-21  train               5.62              1.04
2020-05-22  train               5.53              1.05
2020-05-23  train               6.46              1.11
2020-05-24  train               6.23              0.98
2020-05-25  train               6.78              0.97
2020-05-26  train               6.06              1.05
2020-05-27  train               5.13              1.04
2020-05-28  train               6.51              1.06
2020-05-29  train               6.15              1.05
2020-05-30  train               5.47              1.15
2020-05-31

,mean,min,max
split,,,
test,5.05,4.16,6.35
train,5.37,3.82,6.78
val,5.06,3.88,6.33


In [7]:
for h, d in split_data.items():
    output_path = folder / f"ds_h{h}_split.csv"
    d.to_csv(output_path, index=False)
    print("Saved:", output_path, "Shape:", d.shape)

Saved: /content/drive/MyDrive/Project2_SolarPV/ds_h1_split.csv Shape: (3111, 13)
Saved: /content/drive/MyDrive/Project2_SolarPV/ds_h2_split.csv Shape: (3101, 13)
Saved: /content/drive/MyDrive/Project2_SolarPV/ds_h3_split.csv Shape: (3092, 13)
Saved: /content/drive/MyDrive/Project2_SolarPV/ds_h4_split.csv Shape: (3084, 13)
